# Stage 02: Image Preprocessing

**Status:** Implemented, fully dataset-agnostic, with duplicate-image-folder detection. Deterministic
RGB → Gamma Correction → CLAHE → Processed RGB PNG -- no other transform. See `PROJECT_CODE.md`'s
"Stage 02 Preprocessing Policy" and `PROJECT_STRUCTURE.md`'s Pipeline Overview for the
specification this notebook implements; `SEGMENTATION_ARCHITECTURE.md` for why RGB (not
single-channel) and no resize.

## Objective

Apply `image_preprocessing.py`'s approved Stage 02 pipeline (Gamma Correction, then CLAHE; no
Green Channel Extraction, Ben Graham, Median Filtering, Histogram Equalization, resizing, or
augmentation) to every fundus image in every approved dataset under `datasets/`, writing the
result once to each dataset's `processed/` folder. This notebook does not train anything -- it is
a deterministic transform, not a trained model.

## Dataset discovery -- this notebook contains no dataset names

This notebook discovers what to preprocess at run time, so adding a new dataset (DRIVE,
CHASE_DB1, or anything added later) never requires editing it:

1. Recursively search under `datasets/` (on Google Drive, where the real data lives) for every
   directory that has an immediate `raw/` child -- each one is a preprocessing target (a dataset,
   or a dataset's subtask, e.g. `IDRiD/grading` and `IDRiD/localization` are each their own target
   since each has its own `raw/`).
2. **Skip `EyeQ` completely** -- the one, explicitly-named exception. Stage 1 (Image Quality
   Assessment) is trained on, and must continue to see, the original unprocessed RGB EyeQ images;
   Stage 02 never touches `datasets/EyeQ/`.
3. Within each remaining target's `raw/` subtree, only leaf folders that actually contain fundus
   images (by file extension) become preprocessing candidates -- a folder of label CSVs (e.g.
   IDRiD's `2. Groundtruths/`) or an empty `raw/` (e.g. IDRiD's `segmentation`, not yet populated)
   is never a candidate, so masks/CSVs/labels are never preprocessed and nothing crashes on an
   empty dataset.
4. **Duplicate detection:** every candidate folder's exact set of image *filenames* is compared
   against every other candidate's. Two folders whose filename sets are identical are treated as
   the same underlying image set -- only the alphabetically-first one (by label) is staged and
   preprocessed; every later duplicate is skipped and reported, never staged, never preprocessed.
   This is why `IDRiD/grading` and `IDRiD/localization` -- verified in an earlier session to ship
   byte-identical source photographs under identical filenames -- no longer both get processed:
   `grading` sorts first and is kept, `localization` is detected and skipped as its duplicate,
   automatically, without either name appearing anywhere in this notebook's logic.
5. Output preserves the exact folder hierarchy found under `raw/` for every *kept* folder, mirrored
   under `processed/` -- e.g. `IDRiD/grading/raw/1. Original Images/a. Training Set` →
   `IDRiD/grading/processed/1. Original Images/a. Training Set`. `raw/` itself is only ever read,
   never modified.

**Known, deliberate limitation:** duplicate detection compares filenames only, per this stage's
design (not file content/hashes) -- two folders that coincidentally share an identical filename
set but different image *content* would be misidentified as duplicates. This has not been
observed in any dataset this project uses (the one real case, IDRiD grading/localization, was
independently verified byte-identical by SHA-256 before this behavior was relied upon), but it is
a real edge case a future dataset could in principle trigger.

## Before running

`Runtime > Change runtime type > Hardware accelerator > None` -- this stage is a CPU-bound OpenCV
transform (Gamma LUT + CLAHE), not a trained model; it does not use a GPU.

### Bootstrap

Identical to `stage01_iqa.ipynb`'s Bootstrap cell -- the minimal clone + `sys.path` setup every
stage notebook needs before `colab/common/` is importable.

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

Bootstrap complete: /content/diabetic_retinoplasty


## 1. Setup

`setup.setup()` mounts Google Drive, clones/updates the repository, installs
`requirements.txt`, and enters the repository -- identical call to `stage01_iqa.ipynb`'s Section 1,
reused unmodified.

In [2]:
import setup

setup_info = setup.setup()

Mounted at /content/drive
Repository already present at /content/diabetic_retinoplasty; pulling latest main ...
Dependencies installed.
Entered repository root: /content/diabetic_retinoplasty

Setup complete:
  repo_dir: /content/diabetic_retinoplasty
  drive_mount_point: /content/drive
  env_vars: {'EYEQ_RAW_DIR': '/content/drive/MyDrive/DiabeticRetinopathy/datasets/EyeQ/raw'}
  session_log: /content/drive/MyDrive/DiabeticRetinopathy/logs/setup_2026-08-08_03-57-50.json


## 2. Environment Verification

`verify_environment.verify_all()` checks Python/TensorFlow versions, the repository path, Google
Drive, and required packages -- same call as Stage 1, except **`require_gpu=False`**: Stage 02 is
a CPU-bound OpenCV transform, not a trained model, so it does not need a GPU runtime.

In [3]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=False,
)

[OK] Python version 3.12.13 >= 3.8
[OK] TensorFlow version 2.20.0 >= 2.9.0
[OK] Repository present at /content/diabetic_retinoplasty
[OK] Google Drive mounted at /content/drive
[OK] All packages in /content/diabetic_retinoplasty/requirements.txt are importable
[OK] GPU available: Tesla T4 (1 device(s))
[OK] CUDA version: 12.5.1
Mixed precision enabled: mixed_float16
[OK] Mixed precision policy: mixed_float16


## 3. Dataset Staging

Discovers every preprocessing candidate directly against the Drive-mounted `datasets/` tree (the
only place the real data exists before staging -- a fresh Colab clone never ships dataset
contents, per `PROJECT_CODE.md`'s Training policy), removes exact filename-set duplicates (see the
overview above), then stages only the *kept* candidates' dataset roots onto the local SSD via
`dataset_staging.stage_dataset()` (reused unmodified) and verifies each copy
(`dataset_staging.verify_staged_copy()`). A dataset root whose every candidate folder turns out to
be a duplicate is never staged at all -- not just never preprocessed.

**`SKIP_DATASET_NAMES = {"EyeQ"}` is the one dataset-specific reference this notebook contains,
and it is explicitly required** -- everything else below, including duplicate detection, is driven
entirely by what is found on disk, never by dataset name.

In [4]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
SKIP_DATASET_NAMES = {"EyeQ"}


def discover_preprocessing_targets(datasets_root, skip_names=SKIP_DATASET_NAMES):
    """Recursively finds every directory under `datasets_root` with an immediate
    `raw/` child -- each is a preprocessing target. Skips any target whose
    relative path contains a name in `skip_names`. For each remaining
    target, walks its `raw/` subtree and keeps only leaf folders that
    directly contain fundus images (by extension) -- label/mask/CSV-only
    folders and empty raw/ trees are silently excluded. Returns a list of
    dicts: dataset_label, raw_dir (Drive path), dataset_root (Drive path),
    and image_subfolders (relative path within raw/ -> image count and the
    actual filename set, the latter needed for duplicate detection below).
    """
    targets = []
    for dirpath, dirnames, filenames in os.walk(datasets_root):
        if "raw" not in dirnames:
            continue
        rel_root = os.path.relpath(dirpath, datasets_root)
        path_parts = rel_root.split(os.sep)
        # A "raw" or "processed" folder never contains a real nested dataset root.
        dirnames[:] = [d for d in dirnames if d not in ("raw", "processed")]
        if any(part in skip_names for part in path_parts):
            continue

        raw_dir = os.path.join(dirpath, "raw")
        image_subfolders = []
        for sub_dirpath, _, sub_filenames in os.walk(raw_dir):
            images = sorted(f for f in sub_filenames if f.lower().endswith(IMAGE_EXTENSIONS))
            if not images:
                continue
            rel_to_raw = os.path.relpath(sub_dirpath, raw_dir)
            image_subfolders.append({
                "relative_path": rel_to_raw,
                "image_count": len(images),
                "filenames": frozenset(images),
            })

        if not image_subfolders:
            continue  # has raw/, but no fundus images anywhere in it (e.g. empty, or masks/CSVs only)

        targets.append({
            "dataset_label": rel_root.replace(os.sep, "/"),
            "raw_dir": raw_dir,
            "dataset_root": dirpath,
            "image_subfolders": image_subfolders,
        })
    return targets


def deduplicate_candidates(targets):
    """Flattens every discovered (target, subfolder) pair into one candidate
    list, sorted by label -- so "first discovered" is always the
    alphabetically-first label, deterministic across runs and platforms,
    never dependent on filesystem enumeration order. Two candidates are
    duplicates when their image *filename* sets are exactly equal -- no
    dataset name or path is ever compared, so this works identically for
    any current or future dataset. Returns (unique_candidates,
    duplicate_candidates); each duplicate candidate records the label of
    the earlier candidate its filenames matched.
    """
    candidates = []
    for target in targets:
        for sf in target["image_subfolders"]:
            label = target["dataset_label"] if sf["relative_path"] == "." else (
                target["dataset_label"] + "/" + sf["relative_path"].replace(os.sep, "/")
            )
            candidates.append({
                "label": label,
                "dataset_label": target["dataset_label"],
                "dataset_root": target["dataset_root"],
                "raw_dir": target["raw_dir"],
                "relative_path": sf["relative_path"],
                "filenames": sf["filenames"],
            })
    candidates.sort(key=lambda c: c["label"])

    seen_by_filenames = {}
    unique_candidates, duplicate_candidates = [], []
    for candidate in candidates:
        existing_label = seen_by_filenames.get(candidate["filenames"])
        if existing_label is not None:
            candidate["duplicate_of"] = existing_label
            duplicate_candidates.append(candidate)
        else:
            seen_by_filenames[candidate["filenames"]] = candidate["label"]
            unique_candidates.append(candidate)
    return unique_candidates, duplicate_candidates


discovered_targets = discover_preprocessing_targets(colab_config.DRIVE.datasets_root)
unique_candidates, duplicate_candidates = deduplicate_candidates(discovered_targets)

print(f"Discovered {len(discovered_targets)} dataset target(s), "
      f"{len(unique_candidates) + len(duplicate_candidates)} image folder(s) total:")
for candidate in unique_candidates:
    print(f"  [KEEP] {candidate['label']}  ({len(candidate['filenames'])} images)")
for candidate in duplicate_candidates:
    print(f"  [SKIP -- duplicate of {candidate['duplicate_of']}] {candidate['label']}")

assert "EyeQ" not in [t["dataset_label"] for t in discovered_targets], "EyeQ must never be a target"

Discovered 3 dataset target(s), 6 image folder(s) total:
  [KEEP] APTOS2019/test_images  (1928 images)
  [KEEP] APTOS2019/train_images  (3662 images)
  [KEEP] IDRiD/grading/1. Original Images/a. Training Set  (413 images)
  [KEEP] IDRiD/grading/1. Original Images/b. Testing Set  (103 images)
  [SKIP -- duplicate of IDRiD/grading/1. Original Images/a. Training Set] IDRiD/localization/1. Original Images/a. Training Set
  [SKIP -- duplicate of IDRiD/grading/1. Original Images/b. Testing Set] IDRiD/localization/1. Original Images/b. Testing Set


In [5]:
import os
import shutil

# Only dataset roots with at least one unique image folder need staging.
roots_needed = {
    c["dataset_root"]: (c["dataset_label"], c["raw_dir"])
    for c in unique_candidates
}

for dataset_root, (dataset_label, _raw_dir) in roots_needed.items():
    safe_name = dataset_label.replace("/", "_")

    local_dataset_root = f"/content/datasets/{safe_name}"

    if os.path.exists(local_dataset_root):
        shutil.rmtree(local_dataset_root)

print(
    f"Deleted any incomplete staged datasets. "
    f"{len(roots_needed)} dataset root(s) will be staged."
)

skipped_roots = len(discovered_targets) - len(roots_needed)

if skipped_roots:
    print(
        f"{skipped_roots} dataset root(s) skipped because all image folders were duplicates."
    )

Deleted any incomplete staged datasets. 2 dataset root(s) will be staged.
1 dataset root(s) skipped because all image folders were duplicates.


In [6]:
import dataset_staging

staged_by_root = {}
for dataset_root, (dataset_label, raw_dir) in roots_needed.items():
    safe_name = dataset_label.replace("/", "_")
    staged = dataset_staging.stage_dataset(raw_dir, safe_name)
    dataset_staging.verify_staged_copy(staged)
    staged_by_root[dataset_root] = (safe_name, staged)

# Local-only preprocessing job list -- one entry per *kept* candidate, pointing
# at its staged local path. Duplicate candidates never reach this list, so
# every later section (Verification, Preprocessing, Export, Summary) operates
# only on unique image folders without needing to know duplicates exist.
JOBS = []
for candidate in unique_candidates:
    safe_name, staged = staged_by_root[candidate["dataset_root"]]
    rel = candidate["relative_path"]
    local_raw_dir = staged.local_dir if rel == "." else os.path.join(staged.local_dir, rel)
    processed_suffix = "" if rel == "." else os.sep + rel
    JOBS.append({
        "label": candidate["label"],
        "local_raw_dir": local_raw_dir,
        "local_processed_dir": (
    f"/content/datasets/{safe_name}/processed"
    if rel == "."
    else f"/content/datasets/{safe_name}/processed/{rel}"
),
        "drive_processed_dir": os.path.join(candidate["dataset_root"], "processed")
                                if rel == "." else os.path.join(candidate["dataset_root"], "processed", rel),
    })

print(f"\n{len(JOBS)} preprocessing job(s) staged and ready:")
for job in JOBS:
    print(f"  {job['label']}")
if duplicate_candidates:
    print(f"\n{len(duplicate_candidates)} folder(s) skipped as duplicates (not staged, not preprocessed):")
    for candidate in duplicate_candidates:
        print(f"  {candidate['label']}  -- duplicate of {candidate['duplicate_of']}")

[APTOS2019] staging 5593 files: /content/drive/MyDrive/DiabeticRetinopathy/datasets/APTOS2019/raw -> /content/datasets/APTOS2019 (16 parallel workers) ...
[APTOS2019] staged 5593 files in 279.9s (20 files/s)
[APTOS2019] copy verified: 5593 files, 10216.9 MB match between Drive and local SSD.
[IDRiD_grading] staging 518 files: /content/drive/MyDrive/DiabeticRetinopathy/datasets/IDRiD/grading/raw -> /content/datasets/IDRiD_grading (16 parallel workers) ...
[IDRiD_grading] staged 518 files in 19.2s (27 files/s)
[IDRiD_grading] copy verified: 518 files, 212.3 MB match between Drive and local SSD.

4 preprocessing job(s) staged and ready:
  APTOS2019/test_images
  APTOS2019/train_images
  IDRiD/grading/1. Original Images/a. Training Set
  IDRiD/grading/1. Original Images/b. Testing Set

2 folder(s) skipped as duplicates (not staged, not preprocessed):
  IDRiD/localization/1. Original Images/a. Training Set  -- duplicate of IDRiD/grading/1. Original Images/a. Training Set
  IDRiD/localizatio

## 4. Dataset Verification

Runs `verify_dataset.verify_image_folder()` (reused unmodified -- already dataset-agnostic)
against every staged raw job folder in `JOBS`: directory exists, contains at least one image, and
a random sample decodes without corruption. Duplicate folders were never staged, so they are
never verified either -- there is nothing wasted checking a copy that will never be preprocessed.

In [7]:
import verify_dataset

raw_reports = {job["label"]: verify_dataset.verify_image_folder(job["local_raw_dir"]) for job in JOBS}

[/content/datasets/APTOS2019/test_images] 1928 images | corrupted (of 50 sampled): 0
[/content/datasets/APTOS2019/train_images] 3662 images | corrupted (of 50 sampled): 0
[/content/datasets/IDRiD_grading/1. Original Images/a. Training Set] 413 images | corrupted (of 50 sampled): 0
[/content/datasets/IDRiD_grading/1. Original Images/b. Testing Set] 103 images | corrupted (of 50 sampled): 0


## 5. Preprocessing Configuration

`profile="DR"` (`config.PREPROCESSING_PROFILES.DR`) is Stage 02's approved recipe: Gamma
Correction using `config.PREPROCESSING.DEFAULT_GAMMA`, then CLAHE using
`config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT` / `DEFAULT_CLAHE_TILE_GRID_SIZE`. Identical for
every job in `JOBS` -- nothing about this stage is dataset-specific. Printed below for this run's
own record, since this notebook has no `experiment_manager`-style `metadata.json` (Stage 02 is
not a training run).

In [8]:
import config
from image_preprocessing import preprocess_folder

PROFILE = "DR"

print(f"profile: {PROFILE}")
print(f"gamma: {config.PREPROCESSING.DEFAULT_GAMMA}")
print(f"clahe_clip_limit: {config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT}")
print(f"clahe_tile_grid_size: {config.PREPROCESSING.DEFAULT_CLAHE_TILE_GRID_SIZE}")

profile: DR
gamma: 1.2
clahe_clip_limit: 2.0
clahe_tile_grid_size: (8, 8)


## 6. Preprocessing

Reads and writes entirely on the local SSD for every job in `JOBS`, avoiding the per-file Google
Drive FUSE latency `dataset_staging.py`'s own docstring documents -- Section 8 copies the finished
local output back to Drive in bulk afterward. Uses `image_preprocessing.preprocess_folder()`
unmodified for every job, regardless of which dataset it came from. Each job also writes a
`log_file` CSV (already part of `preprocess_folder()`) as its manifest.

In [9]:
LOG_DIR = "/content/datasets/_logs"
os.makedirs(LOG_DIR, exist_ok=True)

import time

start = time.time()

preprocessing_results = {}

for job in JOBS:
    log_file = os.path.join(LOG_DIR, job["label"].replace("/", "_") + ".csv")

    result = preprocess_folder(
        job["local_raw_dir"],
        job["local_processed_dir"],
        profile=PROFILE,
        log_file=log_file,
    )

    preprocessing_results[job["label"]] = result

    print(
        f"[{job['label']}] "
        f"processed={result.summary.processed} "
        f"skipped={result.summary.skipped} "
        f"failed={result.summary.failed}"
    )

print(f"\nTotal preprocessing time: {(time.time() - start)/60:.2f} minutes")

Preprocessing /content/datasets/APTOS2019/test_images: 100%|██████████| 1928/1928 [04:22<00:00,  7.34it/s]


Processed 1928/1928 images into /content/datasets/APTOS2019/processed/test_images
[APTOS2019/test_images] processed=1928 skipped=0 failed=0


Preprocessing /content/datasets/APTOS2019/train_images: 100%|██████████| 3662/3662 [23:57<00:00,  2.55it/s]


Processed 3662/3662 images into /content/datasets/APTOS2019/processed/train_images
[APTOS2019/train_images] processed=3662 skipped=0 failed=0


Preprocessing /content/datasets/IDRiD_grading/1. Original Images/a. Training Set: 100%|██████████| 413/413 [03:21<00:00,  2.05it/s]


Processed 413/413 images into /content/datasets/IDRiD_grading/processed/1. Original Images/a. Training Set
[IDRiD/grading/1. Original Images/a. Training Set] processed=413 skipped=0 failed=0


Preprocessing /content/datasets/IDRiD_grading/1. Original Images/b. Testing Set: 100%|██████████| 103/103 [00:50<00:00,  2.05it/s]

Processed 103/103 images into /content/datasets/IDRiD_grading/processed/1. Original Images/b. Testing Set
[IDRiD/grading/1. Original Images/b. Testing Set] processed=103 skipped=0 failed=0

Total preprocessing time: 32.53 minutes


## 7. Output Verification

Confirms every job produced zero failures, then re-runs `verify_dataset.verify_image_folder()`
against each local processed folder as an independent check that the written files are
themselves valid, decodable images -- not just that `preprocess_folder()` reported success.

In [10]:
for job in JOBS:
    result = preprocessing_results[job["label"]]
    if result.summary.failed > 0:
        print(f"[WARN] {job['label']}: {result.summary.failed} image(s) failed to preprocess.")
    assert result.summary.total_images == (
        result.summary.processed + result.summary.skipped + result.summary.failed
    ), f"{job['label']}: summary counts do not add up"

processed_reports = {
    job["label"]: verify_dataset.verify_image_folder(job["local_processed_dir"]) for job in JOBS
}

print("\nAll processed output folders verified (see per-folder counts above).")

[/content/datasets/APTOS2019/processed/test_images] 1928 images | corrupted (of 50 sampled): 0
[/content/datasets/APTOS2019/processed/train_images] 3662 images | corrupted (of 50 sampled): 0
[/content/datasets/IDRiD_grading/processed/1. Original Images/a. Training Set] 413 images | corrupted (of 50 sampled): 0
[/content/datasets/IDRiD_grading/processed/1. Original Images/b. Testing Set] 103 images | corrupted (of 50 sampled): 0

All processed output folders verified (see per-folder counts above).


## 8. Export to Google Drive

Bulk-copies each job's finished local `processed/` folder (plus its manifest CSV) up to the
corresponding Drive-mounted `processed/` destination computed during discovery (Section 3) --
once, after processing is complete. The copy helper mirrors `dataset_staging.py`'s own
`_copy_one` / thread-pool pattern for the reverse direction (that module only copies
Drive-to-local) -- the smallest amount of new glue needed to reuse that same proven approach,
without modifying that module.

In [11]:
import os
import shutil
import concurrent.futures
from tqdm.auto import tqdm

def _copy_one_to_drive(src, dst):
    """
    Copy a single file to Google Drive.

    Skip copying if the destination already exists
    and has the same file size.
    """
    os.makedirs(os.path.dirname(dst), exist_ok=True)

    if os.path.exists(dst):
        if os.path.getsize(src) == os.path.getsize(dst):
            return False      # skipped

    shutil.copy2(src, dst)
    return True               # copied


def copy_tree_to_drive(local_dir, drive_dir, max_workers=None):
    """
    Copy an entire directory tree to Drive.

    Returns:
        copied_files
        skipped_files
    """

    if max_workers is None:
        max_workers = min(16, os.cpu_count() or 4)

    file_pairs = []

    for dirpath, _, filenames in os.walk(local_dir):
        rel_dir = os.path.relpath(dirpath, local_dir)
        dest_dir = drive_dir if rel_dir == "." else os.path.join(drive_dir, rel_dir)

        for filename in filenames:
            file_pairs.append((
                os.path.join(dirpath, filename),
                os.path.join(dest_dir, filename),
            ))

    copied = 0
    skipped = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = [
            executor.submit(_copy_one_to_drive, src, dst)
            for src, dst in file_pairs
        ]

        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(futures),
            desc=f"Exporting {os.path.basename(drive_dir)}"
        ):
            if future.result():
                copied += 1
            else:
                skipped += 1

    return copied, skipped


exported_summary = {}

for job in JOBS:

    print(f"\n========== {job['label']} ==========")

    copied, skipped = copy_tree_to_drive(
        job["local_processed_dir"],
        job["drive_processed_dir"]
    )

    log_src = os.path.join(
        LOG_DIR,
        job["label"].replace("/", "_") + ".csv"
    )

    log_dst = os.path.join(
        os.path.dirname(job["drive_processed_dir"]),
        "_logs",
        os.path.basename(log_src)
    )

    _copy_one_to_drive(log_src, log_dst)

    exported_summary[job["label"]] = {
        "copied": copied,
        "skipped": skipped,
    }

    print(
        f"Copied: {copied}    "
        f"Skipped: {skipped}"
    )


print("\n========== Export Summary ==========\n")

total_copied = 0
total_skipped = 0

for name, stats in exported_summary.items():
    total_copied += stats["copied"]
    total_skipped += stats["skipped"]

    print(
        f"{name}\n"
        f"   Copied : {stats['copied']}\n"
        f"   Skipped: {stats['skipped']}\n"
    )

print("------------------------------------")
print(f"Total copied : {total_copied}")
print(f"Total skipped: {total_skipped}")
print("------------------------------------")


========== APTOS2019/test_images ==========


Exporting test_images:   0%|          | 0/1928 [00:00<?, ?it/s]

Copied: 1928    Skipped: 0

========== APTOS2019/train_images ==========


Exporting train_images:   0%|          | 0/3662 [00:00<?, ?it/s]

Copied: 3662    Skipped: 0

========== IDRiD/grading/1. Original Images/a. Training Set ==========


Exporting a. Training Set:   0%|          | 0/413 [00:00<?, ?it/s]

Copied: 413    Skipped: 0

========== IDRiD/grading/1. Original Images/b. Testing Set ==========


Exporting b. Testing Set:   0%|          | 0/103 [00:00<?, ?it/s]

Copied: 103    Skipped: 0

========== Export Summary ==========

APTOS2019/test_images
   Copied : 1928
   Skipped: 0

APTOS2019/train_images
   Copied : 3662
   Skipped: 0

IDRiD/grading/1. Original Images/a. Training Set
   Copied : 413
   Skipped: 0

IDRiD/grading/1. Original Images/b. Testing Set
   Copied : 103
   Skipped: 0

------------------------------------
Total copied : 6106
Total skipped: 0
------------------------------------


## 9. Summary

In [12]:
print("=" * 72)
print("STAGE 02 PREPROCESSING -- RUN SUMMARY")
print("=" * 72)

print(f"Datasets discovered ({len(discovered_targets)}):")
for target in discovered_targets:
    print(f"  • {target['dataset_label']}")

print("\nEyeQ: skipped (Stage 1 only; never processed).\n")

for job in JOBS:
    result = preprocessing_results[job["label"]]
    export = exported_summary[job["label"]]

    print(job["label"])
    print(f"  Raw images        : {raw_reports[job['label']].image_count}")
    print(f"  Processed         : {result.summary.processed}")
    print(f"  Skipped           : {result.summary.skipped}")
    print(f"  Failed            : {result.summary.failed}")
    print(f"  Files copied      : {export['copied']}")
    print(f"  Files skipped     : {export['skipped']}")
    print(f"  Output location   : {job['drive_processed_dir']}")
    print()

if duplicate_candidates:
    print("Duplicate image folders skipped:")
    for candidate in duplicate_candidates:
        print(f"  • {candidate['label']}")
        print(f"      duplicate of {candidate['duplicate_of']}")
    print()

print("-" * 72)
print("Stage 02 completed successfully.")
print("-" * 72)
print("Pipeline implemented:")
print("  RGB Image")
print("      ↓")
print("  Gamma Correction")
print("      ↓")
print("  CLAHE")
print("      ↓")
print("  Processed RGB PNG")
print()

print("Stage 02 characteristics:")
print("  ✓ RGB preserved")
print("  ✓ Native resolution preserved")
print("  ✓ No resize")
print("  ✓ No Green Channel Extraction")
print("  ✓ No Ben Graham preprocessing")
print("  ✓ No Median Filtering")
print("  ✓ No Histogram Equalization")
print("  ✓ No Data Augmentation")
print("  ✓ Dataset-agnostic")
print("  ✓ Duplicate dataset detection")
print()

print("Next stage:")
print("  Stage 03 — Vessel Segmentation")
print("  (implementation begins after the vessel segmentation model is finalized)")

STAGE 02 PREPROCESSING -- RUN SUMMARY
Datasets discovered (3):
  • APTOS2019
  • IDRiD/grading
  • IDRiD/localization

EyeQ: skipped (Stage 1 only; never processed).

APTOS2019/test_images
  Raw images        : 1928
  Processed         : 1928
  Skipped           : 0
  Failed            : 0
  Files copied      : 1928
  Files skipped     : 0
  Output location   : /content/drive/MyDrive/DiabeticRetinopathy/datasets/APTOS2019/processed/test_images

APTOS2019/train_images
  Raw images        : 3662
  Processed         : 3662
  Skipped           : 0
  Failed            : 0
  Files copied      : 3662
  Files skipped     : 0
  Output location   : /content/drive/MyDrive/DiabeticRetinopathy/datasets/APTOS2019/processed/train_images

IDRiD/grading/1. Original Images/a. Training Set
  Raw images        : 413
  Processed         : 413
  Skipped           : 0
  Failed            : 0
  Files copied      : 413
  Files skipped     : 0
  Output location   : /content/drive/MyDrive/DiabeticRetinopathy/dat